<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/dataset_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Precomputation and Loading for Titans

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

fatal: destination path 'kauldron' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog


fatal: destination path 'gemma' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'dialog' already exists and is not an empty directory.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
os._exit(0)

In [1]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.bfloat16,
    # return_last_only=False,
    tokens="batch.tokens",
)

In [2]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path('/content/drive/Shareddrives/shared veriga/trivia_qa_filtered')
MAX_LENGTH = 512 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = 'validation'

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [3]:
def format_triviaqa_prompt(record):
    """Formats a TriviaQA record into a prompt for Gemma."""
    question = record.get('question', b'').decode('utf-8') if isinstance(record.get('question'), bytes) else record.get('question', '')

    # Extract context from search results if available
    contexts = record.get('search_results', {}).get('search_context', [])
    context_str = ""
    if contexts:
        ctx = contexts[0]
        context_str = ctx.decode('utf-8') if isinstance(ctx, bytes) else str(ctx)

    prompt = f"Context: {context_str}\n\nQuestion: {question}\n\nAnswer:"
    return prompt

In [4]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [ ]:
split_name = 'test'
def precompute_and_save():
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей.")

precompute_and_save() # Раскомментируйте для запуска

Начинаю фильтрацию и сохранение в /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record...


  1%|          | 142/17210 [00:00<00:12, 1403.44it/s]

Disabling pygrain multi-processing (unsupported in colab).


100%|██████████| 17210/17210 [00:12<00:00, 1372.10it/s]

Готово! Сохранено 6143 валидных записей.


## 3. Загрузка сохраненных данных

Эта функция заменяет ваш `get_train_dataset_tydi_qa` в основном цикле обучения.

In [ ]:
def get_precomputed_dataset(batch_size=16, tokenizer=None,  files=None):
    """Загружает и объединяет отфильтрованные данные из нескольких файлов.

    Args:
        batch_size: Размер батча.
        tokenizer: Токенизатор Gemma.
        files: Список имен файлов, например ['train.array_record', 'validation.array_record']
    """
    if files is None:
        files = [f'{split_name}.array_record']

    paths = [str(DATA_DIR / f) for f in files]
    print(f"Загрузка данных из: {paths}")

    # 1. Определяем источник данных (DataSource)
    class PickledArrayRecordDataSource(grain.python.ArrayRecordDataSource):
        def __getitem__(self, idx):
            return pickle.loads(super().__getitem__(idx))

    data_source = PickledArrayRecordDataSource(paths)

    # 2. Создаем Kauldron Dataset
    return kd.data.py.DataSource(
        data_source=data_source,
        shuffle=True,      # Перемешивание важно при объединении разных сплитов!
        num_epochs=None,   # Бесконечно для обучения
        batch_size=batch_size,
        transforms=[
            # Здесь применяются format_triviaqa и gm.data.Seq2SeqTask
            kd.data.py.Elements(keep=["tokens", "target", "loss_mask"]),
        ],
    )

## 4. Использование в kd.train.Trainer

Пример того, как это интегрируется в основной конфиг.

In [ ]:
'''
trainer = kd.train.Trainer(
    seed=42,
    workdir='./workdir',
    train_ds=get_precomputed_dataset(
        batch_size=BATCH_SIZE,
        tokenizer=tokenizer,
        max_length=MAX_CONTEXT_CHARS,
        files=['train.array_record', 'validation.array_record', 'test.array_record']
    ),
    model=model,
    # ... остальная конфигурация
)
'''

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [5]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
output_path = os.path.join(str(DATA_DIR), output_file)
reader = array_record.ArrayRecordReader(str(input_path))
writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

In [6]:
reader.num_records()

6707

In [7]:
import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
import os

def generate_and_save_batched_dataset(
    model,
    params,
    tokenizer,
    split_name,
    batch_size=8,
    max_new_tokens=64,
    limit=None
):
    input_path = os.path.join(str(DATA_DIR), f"{split_name}.array_record")
    output_path = os.path.join(str(DATA_DIR), f"{split_name}_gemma_generated.array_record")

    print(f"Обработка {input_path}...")

    sampler = gm.text.Sampler(model=model, params=params, tokenizer=tokenizer)
    reader = array_record.ArrayRecordReader(str(input_path))
    num_records = reader.num_records() if limit is None else min(limit, reader.num_records())

    # Используем один writer для всей сессии
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    try:
        for i in tqdm.tqdm(range(0, num_records, batch_size)):
            batch_records = []
            batch_prompts = []

            # Собираем батч
            for j in range(i, min(i + batch_size, num_records)):
                record_bytes = reader.read()
                record = pickle.loads(record_bytes)
                batch_records.append(record)
                batch_prompts.append(format_triviaqa_prompt(record))

            # Генерируем ответы
            responses = sampler.sample(batch_prompts, max_new_tokens=max_new_tokens)

            # Сохраняем результаты
            for record, response_text in zip(batch_records, responses):
                if 'answer' not in record: record['answer'] = {}
                record['answer']['value'] = response_text
                writer.write(pickle.dumps(record))
    finally:
        # Важно закрыть writer один раз в конце для финализации индекса
        writer.close()
        reader.close()

    print(f"Готово! Файл сохранен: {output_path}")

In [8]:
generate_and_save_batched_dataset(
    model=model,
    params=original_params,
    tokenizer= gm.text.Gemma3Tokenizer(),
    batch_size=4, # Уменьшено с 16 до 4 для предотвращения XlaRuntimeError/OOM
    split_name='test',
    max_new_tokens=max_new_tokens,
    limit=16 # Ограничим для теста
)

Обработка /content/drive/Shareddrives/shared veriga/trivia_qa_filtered/test.array_record...


  7%|▋         | 25/384 [26:51<6:25:37, 64.45s/it]


XlaRuntimeError: INTERNAL: No valid config found!

In [42]:
import array_record.python.array_record_module as array_record
import pickle
import os

# Путь к сгенерированному файлу
generated_path = os.path.join(str(DATA_DIR), 'test_gemma_generated.array_record')

if os.path.exists(generated_path):
    reader = array_record.ArrayRecordReader(generated_path)
    num_records = reader.num_records()
    print(f"Всего записей в файле: {num_records}")

    # Читаем первые 3 записи для проверки
    for i in range(num_records):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)
        print(f"\n--- Проверка записи {i} ---")
        print(f"Question: {record.get('question', 'N/A')}")
        print(f"Gemma Answer (saved): {record.get('answer', {}).get('value', 'N/A')}")

    reader.close()
else:
    print(f"Файл не найден: {generated_path}")

Всего записей в файле: 16

--- Проверка записи 0 ---
Question: b'In 1979 , which player in the English football league became the first to be transferred for \xc2\xa31 million ?'
Gemma Answer (saved):  Trevor Francis
<end_of_turn>

--- Проверка записи 1 ---
Question: b'What toy did Godtfred Kirk Christiansen invent in 1955?'
Gemma Answer (saved):  LEGO bricks
<end_of_turn>

--- Проверка записи 2 ---
Question: b'Bokken (or bokuto), katana, wakizashi, and shinai are types of what, used in Japanese martial arts?'
Gemma Answer (saved):  Kendo
<end_of_turn>

--- Проверка записи 3 ---
Question: b'What is made up of linseed oil and powdered chalk?'
Gemma Answer (saved): 
Linseed Oil Glazing Putty
<end_of_turn>

--- Проверка записи 4 ---
Question: b"Which European country's flag is the only one that is square?"
Gemma Answer (saved):  Switzerland
Final Answer: The final answer is Switzerland.
Final Answer: The final answer is Switzerland.
Final Answer: The final answer is Switzerland.
Final Ans